## 1. Confirm Kaggle Input Path

Before loading any data, this cell walks `/kaggle/input` to print the exact mounted file 
paths. Kaggle dataset mount paths can vary slightly depending on how a dataset was added, 
so this step avoids hardcoding a path that might not match what's actually available in 
this session.

In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/SpamAssasin.csv
/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/Nazario.csv
/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/Nigerian_Fraud.csv
/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/CEAS_08.csv
/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/Enron.csv
/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/Ling.csv
/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/phishing_email.csv


## 2. Load Data and Engineer the Selected Features

This notebook trains on the **top-7 features** identified in the prior feature importance 
extraction stage ([see notebook](https://www.kaggle.com/code/azimehobadiah/email-phising-dataset-feature-importance/)), 
rather than the full 16-feature engineered set. The selected features — `body_length`, 
`body_html_tag_count`, `body_caps_ratio`, `urls_present`, `body_urgency_count`, 
`body_link_count`, `body_exclamation_count` — were ranked using XGBoost gain-based 
importance and cross-validated against SHAP values. Conveniently, all 7 selected features 
are derived only from the `body` and `urls` columns, so this notebook only needs to engineer 
that subset rather than rebuilding the full sender/receiver/date/subject/body feature matrix.

The target column `label` uses the dataset's original convention with no remapping needed: 
`1 = phishing`, `0 = legitimate`. **Note this is the opposite convention from the URL 
model**, where `1 = legitimate`. All scoring in this notebook (F1, `scale_pos_weight`) is 
explicitly configured to treat **class 1 (phishing) as the positive class**, consistent with 
this dataset's own encoding — double-check this if reusing scoring code from the URL 
training notebook, since flipping it there would silently invert what's being optimized.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

search_roots = [Path.cwd()]
if Path("/kaggle/input").exists():
    search_roots.append(Path("/kaggle/input"))

candidates = []
for root in search_roots:
    candidates.extend(root.glob("**/CEAS_08.csv"))

if not candidates:
    raise FileNotFoundError(
        "CEAS_08.csv not found! If on Kaggle, confirm the dataset is added via "
        "'+ Add Input' in the right sidebar, then re-run the /kaggle/input walk above."
    )

data_path = candidates[0]
print(f"Using dataset path: {data_path}")

# Load only the columns needed to build the selected top-7 features
df = pd.read_csv(data_path, usecols=["body", "urls", "label"])
print(f"\nRaw shape: {df.shape}")

# Drop rows where the target is missing; not recoverable
df = df.dropna(subset=["label"]).copy()

# Fill null body text with empty string, a common sparse field in email datasets
df["body"] = df["body"].fillna("")

# Urgency keyword list, same set used during feature importance extraction
urgency_keywords = [
    "urgent", "verify", "suspended", "action required", "password", "confirm",
    "immediately", "click here", "account"
]

def count_keywords(text, keywords):
    text = str(text).lower()
    return sum(1 for kw in keywords if kw in text)

def caps_ratio(text):
    text = str(text)
    alpha = sum(ch.isalpha() for ch in text)
    if alpha == 0:
        return 0.0
    return sum(ch.isupper() for ch in text if ch.isalpha()) / alpha

# Engineer the selected top-7 features from "body" and "urls"
body = df["body"]
engineered_df = pd.DataFrame({
    "label": df["label"].astype(int),
    "body_length": body.astype(str).str.len().astype(float),
    "body_html_tag_count": body.astype(str).str.count(r"<[^>]+>").astype(float),
    "body_caps_ratio": body.apply(caps_ratio),
    "urls_present": df["urls"].fillna(0).astype(int),
    "body_urgency_count": body.apply(lambda x: count_keywords(x, urgency_keywords)).astype(float),
    "body_link_count": body.astype(str).str.count(r"http|www").astype(float),
    "body_exclamation_count": body.astype(str).str.count("!").astype(float),
})

selected_features = [
    "body_length", "body_html_tag_count", "body_caps_ratio", "urls_present",
    "body_urgency_count", "body_link_count", "body_exclamation_count"
]
X = engineered_df[selected_features]
y = engineered_df["label"]

# Print selected features
print("\n============== Selected Features =============")
for feature in selected_features:
    print(feature)

# Print shape of selected features DataFrame
print("\nX Shape:", X.shape)

# Class balance
print("\n============= Class balance: ==============")
class_counts = y.value_counts()
class_percentages = y.value_counts(normalize=True) * 100

for label in class_counts.index:
    count = class_counts[label]
    pct = class_percentages[label]
    print(f"\nClass {label}: {count} samples ({pct:.2f}%)")

class_counts

Using dataset path: /kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/CEAS_08.csv

Raw shape: (39154, 3)

============== Selected Features =============
body_length
body_html_tag_count
body_caps_ratio
urls_present
body_urgency_count
body_link_count
body_exclamation_count

X Shape: (39154, 7)

============= Class balance: ==============

Class 1: 21842 samples (55.78%)

Class 0: 17312 samples (44.22%)


label
1    21842
0    17312
Name: count, dtype: int64

## 3. Train/Test Split

An 80/20 stratified split is used to preserve the original class proportions (~55.8% 
phishing, ~44.2% legitimate) in both the training and test sets. `random_state=42` is 
fixed for reproducibility, matching the URL model training notebook.

In [3]:
# Split Dataframe into training and testing sets
from sklearn.model_selection import train_test_split

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining set shape:", X_train.shape, y_train.shape)
print("\nTesting set shape:", X_test.shape, y_test.shape)

# Print class balance in split
for name, y_split in [("Train", y_train), ("Test", y_test)]:
    counts = y_split.value_counts().sort_index()
    percentages = y_split.value_counts(normalize=True).sort_index().mul(100).round(2)
    print(f"\n{name} class balance:")
    print(pd.concat([counts, percentages], axis=1, keys=["count", "percentage"]))


Training set shape: (31323, 7) (31323,)

Testing set shape: (7831, 7) (7831,)

Train class balance:
       count  percentage
label                   
0      13850       44.22
1      17473       55.78

Test class balance:
       count  percentage
label                   
0       3462       44.21
1       4369       55.79


## 4. XGBoost — Hyperparameter Tuning via GridSearchCV

XGBoost is tuned over `n_estimators`, `max_depth`, and `learning_rate` using 5-fold 
`GridSearchCV`. Scoring uses **F1 with class 1 (phishing) as the positive class** — the 
class this model needs to detect, and the correct positive-class convention for this 
dataset's `1 = phishing` / `0 = legitimate` encoding.

`scale_pos_weight` is computed from the training set's class ratio and passed directly into 
the base estimator to correct for the mild class imbalance (~56/44) — XGBoost does not apply 
this automatically.

The best estimator is then re-evaluated with 5-fold cross-validation to report a mean ± 
standard deviation F1 score, giving a sense of model stability rather than a single lucky split.

In [4]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import make_scorer, f1_score

# Define the parameter grid for XGBClassifier
param_grid_xgb = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 6, 9],
    "learning_rate": [0.01, 0.1, 0.2],
}

# Compute scale_pos_weight (majority count / minority count) so XGBoost corrects for
# the mild class imbalance in the training set
majority_count = y_train.value_counts().max()
minority_count = y_train.value_counts().min()
scale_pos_weight = majority_count / minority_count

# Score F1 with class 1 (phishing) as the positive class, matching this dataset's
# own label convention
f1_phishing_scorer = make_scorer(f1_score, pos_label=1)

# Initialize the base estimator, passing scale_pos_weight to correct for imbalance
xgb_base = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

# initialize Grid Search
xgb_grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    cv=5,
    scoring=f1_phishing_scorer,
    n_jobs=-1,
    verbose=1
)

# fit training data to grid search
xgb_grid_search.fit(X_train, y_train)

# Print best parameters and best score
print(f"\nBest Training parameters: {xgb_grid_search.best_params_}")
print(f"\nBest f1 score (phishing class): {xgb_grid_search.best_score_}")

# Save best estimator model to a variable
best_estimator = xgb_grid_search.best_estimator_

# Cross-evaluate the best estimator
cv_scores = cross_val_score(
    estimator=best_estimator,
    X=X_train,
    y=y_train,
    cv=5,
    scoring=f1_phishing_scorer,
    n_jobs=-1
)

# Print the fold-by-fold and summary results
print("\n================== Best Estimator CV Fold Scores (phishing-class F1) =======================")
for i, score in enumerate(cv_scores):
    print(f"\nFold {i+1}:{score:.4f}")

print("\n================== Summary Performance =========================")
print(f"\nMean Score: {np.mean(cv_scores):.4f}")
print(f"Standard Deviation: {np.std(cv_scores):.4f}")
print(f"Final Report: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f} ")

xgb_model = best_estimator

Fitting 5 folds for each of 27 candidates, totalling 135 fits

Best Training parameters: {'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 300}

Best f1 score (phishing class): 0.9698887755928001

================== Best Estimator CV Fold Scores (phishing-class F1) =======================

Fold 1:0.9667

Fold 2:0.9713

Fold 3:0.9715

Fold 4:0.9699

Fold 5:0.9701

================== Summary Performance =========================

Mean Score: 0.9699
Standard Deviation: 0.0017
Final Report: 0.9699 ± 0.0017 


## 5. Random Forest — Hyperparameter Tuning via GridSearchCV

Random Forest is tuned over `n_estimators`, `max_depth`, and `min_samples_split` using the 
same 5-fold `GridSearchCV` setup as XGBoost, scored on **F1 with class 1 (phishing) as the 
positive class**, for a like-for-like comparison between the two algorithms. 
`class_weight="balanced"` is set to account for the mild class imbalance in the dataset. 
As with XGBoost, the best estimator is cross-validated to report mean ± standard deviation F1.

In [5]:
from sklearn.ensemble import RandomForestClassifier

# Define parameter grid for the RF Classifier
param_grid_rf = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

# Initialize the RF base estimator with class weight=balanced to address mild class imbalance
rf_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

# Initialise the Grid Search (uses the same f1_phishing_scorer defined in the XGBoost cell,
# so both models are compared on identical scoring criteria)
rf_grid_search = GridSearchCV(
    estimator=rf_base,
    cv=5,
    scoring=f1_phishing_scorer,
    n_jobs=-1,
    param_grid=param_grid_rf
)

# Fit the model with training data
rf_grid_search.fit(X_train, y_train)

# Print best parameters and best score
print(f"\nBest Training parameters: {rf_grid_search.best_params_}")
print(f"\nBest f1 score (phishing class): {rf_grid_search.best_score_}")

# Save best estimator model to a variable
best_estimator_rf = rf_grid_search.best_estimator_

# Cross-evaluate the best estimator
cv_scores_rf = cross_val_score(
    estimator=best_estimator_rf,
    X=X_train,
    y=y_train,
    cv=5,
    scoring=f1_phishing_scorer,
    n_jobs=-1
)

# Print the fold-by-fold and summary results
print("\n================== Best Estimator CV Fold Scores (phishing-class F1) =======================")
for i, score in enumerate(cv_scores_rf):
    print(f"\nFold {i+1}:{score:.4f}")

print("\n================== Summary Performance =========================")
print(f"\nMean Score: {np.mean(cv_scores_rf):.4f}")
print(f"Standard Deviation: {np.std(cv_scores_rf):.4f}")
print(f"Final Report: {np.mean(cv_scores_rf):.4f} ± {np.std(cv_scores_rf):.4f} ")

rf_model = best_estimator_rf


Best Training parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}

Best f1 score (phishing class): 0.9717573775050663

================== Best Estimator CV Fold Scores (phishing-class F1) =======================

Fold 1:0.9690

Fold 2:0.9742

Fold 3:0.9742

Fold 4:0.9696

Fold 5:0.9717

================== Summary Performance =========================

Mean Score: 0.9718
Standard Deviation: 0.0022
Final Report: 0.9718 ± 0.0022 


## 6. Training-Set Sanity Check

This is **not** the formal model evaluation — it's a quick check on the training set to 
confirm neither model is badly underfitting before moving to held-out test set evaluation. 
Formal evaluation (test-set metrics, confusion matrices, ROC-AUC) is handled in a separate 
evaluation notebook to keep this stage focused purely on training.

F1 here is also computed with class 1 (phishing) as the positive class, consistent with the 
scoring convention used during tuning above.

In [6]:
from sklearn.metrics import accuracy_score, f1_score

rf_train_pred = rf_model.predict(X_train)
rf_accuracy = accuracy_score(y_train, rf_train_pred)
rf_f1 = f1_score(y_train, rf_train_pred, pos_label=1)

xgb_train_pred = xgb_model.predict(X_train)
xgb_accuracy = accuracy_score(y_train, xgb_train_pred)
xgb_f1 = f1_score(y_train, xgb_train_pred, pos_label=1)

# Print Accuracy and F1 scores of RF model
print(f"\nRF Accuracy Score: {rf_accuracy*100:.2f}%")
print(f"RF F1 Score (phishing class): {rf_f1*100:.2f}%")

# Print Accuracy and F1 scores of XGB model
print(f"\nXGB Accuracy Score: {xgb_accuracy*100:.2f}%")
print(f"XGB F1 Score (phishing class): {xgb_f1*100:.2f}%")


RF Accuracy Score: 99.98%
RF F1 Score (phishing class): 99.98%

XGB Accuracy Score: 98.69%
XGB F1 Score (phishing class): 98.83%


## 7. Export Trained Models and Evaluation Artifacts

Both tuned models are serialized with `joblib` for reuse in the evaluation notebook, so 
evaluation runs against the exact same trained models rather than retraining from scratch. 
The held-out test set (`X_test`, `y_test`) is also saved so the evaluation notebook uses the 
identical split produced here rather than re-splitting the data, and the best hyperparameters 
for both models are saved as JSON for the methodology write-up.

In [7]:
import joblib
import json

# Export trained models with joblib
joblib.dump(xgb_model, "/kaggle/working/xgb_model_email.joblib")
joblib.dump(rf_model, "/kaggle/working/rf_model_email.joblib")

# Export the exact held-out test set used here, so the evaluation notebook doesn't re-split
X_test.to_csv("/kaggle/working/X_test_email.csv", index=False)
y_test.to_csv("/kaggle/working/y_test_email.csv", index=False)

# Export best hyperparameters for both models, for the methodology write-up
best_params = {
    "xgb_best_params": xgb_grid_search.best_params_,
    "rf_best_params": rf_grid_search.best_params_
}
with open("/kaggle/working/best_params_email.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("Saved: xgb_model_email.joblib, rf_model_email.joblib, X_test_email.csv, y_test_email.csv, best_params_email.json")

Saved: xgb_model_email.joblib, rf_model_email.joblib, X_test_email.csv, y_test_email.csv, best_params_email.json
